In [0]:
#Importing Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.fs.rm(DELTA_PATH, recurse=True)

True

**Step 1:Loading the Data**

In [0]:
DELTA_PATH = "/Volumes/workspace/default/delta_volume/Superstore_delta"
CSV_PATH = "/Volumes/workspace/default/delta_volume/Superstore.csv"

def load_csv_to_delta(csv_path: str, delta_path: str):

    # Read CSV
    df = (
        spark.read.format("csv").option("header", "true").option("inferSchema", "true").option("escape", '"').load(csv_path)
    )

    # Replace spaces in column names
    df = df.toDF(*[c.replace(" ", "_") for c in df.columns])

    # Check the data

    # Write to Delta
    (
        df.write.format("delta").mode("overwrite").save(delta_path)
    )

    print(f"Loaded {df.count()} rows into Delta table at {delta_path}")
    return df


df = load_csv_to_delta(CSV_PATH, DELTA_PATH)

Loaded 9994 rows into Delta table at /Volumes/workspace/default/delta_volume/Superstore_delta


** Step 2:Cleaning

In [0]:
def clean_dataframe(df):
   
    print(f'No.of rows in file contains :{df.count()}')

    df = df.dropDuplicates()
    df = df.filter(col("Row_ID").isNotNull())
    df = df.fillna(0,       subset=["Sales", "Quantity", "Discount", "Profit"])
    df = df.fillna("Unknown", subset=["Customer_Name", "City", "State", "Region"])

    print(f'No. of rows after cleaning :{df.count()}')
    
    return df

df_clean = clean_dataframe(df)


No.of rows in file contains :9994
No. of rows after cleaning :9994


**Step 3:New/Incremental Data**

In [0]:
def create_incremental_data(spark):
   
    incremental_data = [
        # UPDATES — existing Row_IDs with changed values
        (1, "CA-2016-152156", "11/8/2016", "11/11/2016", "Second Class",
         "CG-12520", "Claire Gute", "Consumer", "United States",
         "Henderson", "Kentucky", "42420", "South",
         "FUR-BO-10001798", "Furniture", "Bookcases",
         "Bush Somerset Collection Bookcase", 300.00, 2, 0.0, 55.00),
        (2, "CA-2016-152156", "11/8/2016", "11/11/2016", "Second Class",
         "CG-12520", "Claire Gute", "Consumer", "United States",
         "Henderson", "Kentucky", "42420", "South",
         "FUR-CH-10000454", "Furniture", "Chairs",
         "Hon Deluxe Fabric Chair", 800.00, 3, 0.0, 250.00),
        # INSERTS — new Row_IDs not in base table
        (9995, "CA-2024-999001", "7/1/2024", "7/5/2024", "First Class",
         "HJ-99001", "Heeral Jain", "Corporate", "United States",
         "Jaipur", "Rajasthan", "302001", "South",
         "TEC-PH-10004977", "Technology", "Phones",
         "Apple iPhone 15 Pro", 1299.99, 1, 0.0, 399.99),
        (9996, "CA-2024-999002", "7/2/2024", "7/6/2024", "Standard Class",
         "HJ-99002", "Celebal Tech", "Corporate", "United States",
         "Jaipur", "Rajasthan", "302001", "South",
         "OFF-PA-10002365", "Office Supplies", "Paper",
         "Avery Premium Paper Ream", 45.99, 10, 0.05, 12.50),
    ]
    columns = [
        "Row_ID","Order_ID","Order_Date","Ship_Date","Ship_Mode",
        "Customer_ID","Customer_Name","Segment","Country","City",
        "State","Postal_Code","Region","Product_ID","Category",
        "Sub-Category","Product_Name","Sales","Quantity","Discount","Profit"
    ]
    df_incremental = spark.createDataFrame(incremental_data, schema=columns)
    print(f"Incremental records: {df_incremental.count()} )")
    df_incremental.show()
    return df_incremental

    df_incremental = create_incremental_data(spark)
    df_incremental.write.mode("append").option("header", "true").csv("/Volumes/workspace/default/delta_volume/Superstore_incremental.csv")

__Path of the Delta Table__

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/delta_volume/Superstore_delta"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/delta_volume/Superstore_delta/_delta_log/,_delta_log/,0,1783283014532
dbfs:/Volumes/workspace/default/delta_volume/Superstore_delta/part-00000-28f4e06b-0aeb-4a48-80b6-e6f17baf53d6.c000.snappy.parquet,part-00000-28f4e06b-0aeb-4a48-80b6-e6f17baf53d6.c000.snappy.parquet,429572,1783283011000


**Step 4:Using Upsert/MERGE** 

In [0]:
def merge_incremental_data(delta_path: str, df_incremental) -> None:
   
    delta_table = DeltaTable.forPath(spark, delta_path)

    (
        delta_table.alias("target")
        .merge(
            df_incremental.alias("source"),
            "target.Row_ID = source.Row_ID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGE completed successfully")


**Step 5:Validation**

In [0]:
def validate_delta_table(delta_path: str) -> None:
   
    
    df_final = spark.read.format("delta").load(delta_path)

    total_rows  = df_final.count()
    # Verify updated rows
    print("UPDATED rows:")
    df_final.filter(col("Row_ID").isin([1, 2])).select(
        "Row_ID", "Customer_Name", "Sales", "Profit"
    ).show()

    # Verify inserted rows
    print("INSERTED rows :")
    df_final.filter(col("Row_ID").isin([9995, 9996])).select(
        "Row_ID", "Customer_Name", "Product_Name", "Sales"
    ).show()

validate_delta_table(DELTA_PATH)

UPDATED rows:
+------+-------------+------+-------+
|Row_ID|Customer_Name| Sales| Profit|
+------+-------------+------+-------+
|     1|  Claire Gute|261.96|41.9136|
|     2|  Claire Gute|731.94|219.582|
+------+-------------+------+-------+

INSERTED rows :
+------+-------------+------------+-----+
|Row_ID|Customer_Name|Product_Name|Sales|
+------+-------------+------------+-----+
+------+-------------+------------+-----+



In [0]:
def show_delta_history(delta_path: str) -> None:
   
    delta_table = DeltaTable.forPath(spark, delta_path)
    print("Delta Lake Transaction History:")
    delta_table.history().select(
        "version", "timestamp", "operation", "operationParameters"
    ).show(truncate=False)

show_delta_history(DELTA_PATH)

Delta Lake Transaction History:
+-------+-------------------+---------+------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                         |
+-------+-------------------+---------+------------------------------------------------------------+
|0      |2026-07-05 20:23:32|WRITE    |{mode -> Overwrite, statsOnLoad -> false, partitionBy -> []}|
+-------+-------------------+---------+------------------------------------------------------------+



In [0]:
def display_final_summary(delta_path: str) -> None:
    
    df_final = spark.read.format("delta").load(delta_path)

    print("Final Delta Table — Sample rows:")
    df_final.show(10)

    print("Region-wise Total Sales:")
    (
        df_final
        .groupBy("Region")
        .agg({"Sales": "sum", "Profit": "sum", "Row_ID": "count"})
        .withColumnRenamed("sum(Sales)",  "Total_Sales")
        .withColumnRenamed("sum(Profit)", "Total_Profit")
        .withColumnRenamed("count(Row_ID)", "Order_Count")
        .orderBy("Total_Sales", ascending=False)
        .show()
    )

    print("Category-wise Order Count:")
    (
        df_final.groupBy("Category") .count().orderBy("count").show()
    )

display_final_summary(DELTA_PATH)

Final Delta Table — Sample rows:
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furn

## Summary

| Step | Operation | Method |
|---|---|---|
| 1 | Load CSV to Delta | `spark.read.csv` → `write.format('delta')` |
| 2 | Clean data | `dropDuplicates()`, `filter(isNotNull)`, `fillna()` |
| 3 | incremental data | `spark.createDataFrame()`  |
| 4 | MERGE (upsert) | `DeltaTable.merge().whenMatchedUpdateAll().whenNotMatchedInsertAll()` |
| 5 | Validate | Row count, duplicate check, null check|
| 6 | Time travel | `DeltaTable.history()` — versioned transactions |
| 7 | Final summary | GroupBy aggregations on final state |

### Key Insights
- **Delta Lake MERGE** is the industry-standard pattern for incremental data loading — avoids full table rewrites
- **Time Travel** allows reading any previous version: `spark.read.format('delta').option('versionAsOf', 0).load(path)`
- **SCD Type 1**  — matched records are overwritten.No need to keep history. 
- **SCD Type 2** -updates old records .History is kept
- **_delta_log** directory stores all transaction metadata as JSON — this is what enables versioning and recovery